# 03 — Construcción de OBT (Snowpark API)

Utiliza el flujo de **Snowpark DataFrames** para calcular derivadas complejas asegurando la ejecución dentro del motor de Snowflake.

In [ ]:
import os
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, to_date, call_builtin, when, lit, round, dayofweek

print("=" * 60)
print("CELDA 1: Inicialización")
print("=" * 60)

connection_parameters = {
    "account":     os.environ['SF_ACCOUNT'],
    "user":        os.environ["SF_USER"],
    "password":    os.environ["SF_PASSWORD"],
    "database":    os.environ["SF_DATABASE"],
    "schema":      os.environ["SF_SCHEMA_RAW"],
    "warehouse":   os.environ["SF_WAREHOUSE"],
    "role":        os.environ["SF_ROLE"],
}

session = Session.builder.configs(connection_parameters).create()

print("✓ Snowpark Session Inicializada.")

## 3.1 Construcción del DAG de la OBT

In [ ]:
print("=" * 60)
print("CELDA 2: Extracción de Fechas y Derivadas (Snowpark API)")
print("=" * 60)

# Cargar tabla enriquecida del paso anterior
df_enriched = session.table("TRIPS_ENRICHED")

# Lógica del OBT
df_obt = (
    df_enriched
    # Componentes de Fecha/Hora
    .with_column("pickup_date", to_date(col("pickup_datetime")))
    .with_column("pickup_hour", call_builtin("DATE_PART", 'hour', col("pickup_datetime")))
    .with_column("dropoff_date", to_date(col("dropoff_datetime")))
    .with_column("dropoff_hour", call_builtin("DATE_PART", 'hour', col("dropoff_datetime")))
    .with_column("day_of_week", dayofweek(col("pickup_datetime")))
    .with_column("month", col("source_month"))
    .with_column("year", col("source_year"))
    
    # Renombrar servicio para lineage
    .with_column("source_service", col("service_type"))
    
    # Derivadas Numéricas (Usamos TIMESTAMPDIFF nativo de Snowflake via call_builtin)
    .with_column("trip_duration_min", 
                call_builtin("TIMESTAMPDIFF", 'minute', col("pickup_datetime"), col("dropoff_datetime")))
    
    .with_column("avg_speed_mph", 
                when((col("trip_duration_min") > 0) & (col("trip_distance") > 0),
                       round(col("trip_distance") / (col("trip_duration_min") / 60.0), 2))
                 .otherwise(lit(None)))
                 
    .with_column("tip_pct", 
                when(col("fare_amount") > 0,
                       round((col("tip_amount") / col("fare_amount")) * 100, 2))
                 .otherwise(lit(None)))
)

print("✓ DAG para OBT construido.")

## 3.2 Materialización
Guardamos la tabla OBT_TRIPS en el esquema ANALYTICS de Snowflake.

In [ ]:
print("=" * 60)
print("CELDA 3: Escritura en ANALYTICS")
print("=" * 60)

schema_analytics = os.environ["SF_SCHEMA_ANALYTICS"]
table_full_name = f"{schema_analytics}.OBT_TRIPS"

print(f"\n>>> Escribiendo {table_full_name} en Snowflake...")
df_obt.write.mode("overwrite").save_as_table(table_full_name)

print("✓ OBT_TRIPS materializada con éxito.")
print("=" * 60)

## 3.3 Verificación y Conteo

In [ ]:
print("=" * 60)
print("CELDA 4: Conteo por servicio en OBT_TRIPS")
print("=" * 60)

df_final = session.table(f"{os.environ['SF_SCHEMA_ANALYTICS']}.OBT_TRIPS")

df_final.group_by("service_type").count().sort("service_type").show()

print("=" * 60)